# Load processed v3 splits

This notebook loads the pre-extracted train/validation/test splits used throughout the thesis. The original WRDS extraction is documented in `REPRODUCIBILITY.md` under "Data provenance" and is not re-run here. A reviewer with WRDS access can regenerate the splits by following those steps.


# Import packages


In [ ]:
import pandas as pd


# Load train / validation / test splits from files/


In [ ]:
train_df = pd.read_csv('files/train_data_v3.csv.gz')
val_df = pd.read_csv('files/val_data_v3.csv.gz')
test_df = pd.read_csv('files/test_data_v3.csv.gz')


# Universe and split summary


In [ ]:
# 47-stock universe = intersection of full-coverage PERMNOs in train and val (sprint5_walkforward.py logic)
train_dates = sorted(train_df['date'].unique())
train_cov = train_df.groupby('PERMNO')['date'].nunique()
full_train_stocks = set(train_cov[train_cov == len(train_dates)].index)
val_dates = sorted(val_df['date'].unique())
val_cov = val_df.groupby('PERMNO')['date'].nunique()
full_val_stocks = set(val_cov[val_cov == len(val_dates)].index)
universe_47 = sorted(full_train_stocks & full_val_stocks)
assert len(universe_47) == 47, f"Expected 47 stocks, got {len(universe_47)}"

for name, df in [('Train', train_df), ('Val', val_df), ('Test', test_df)]:
    print(f"{name}: {len(df)} rows, {df['PERMNO'].nunique()} PERMNOs, "
          f"{df['date'].nunique()} trading days, "
          f"{df['date'].min()} to {df['date'].max()}")

universe_df = (train_df[train_df['PERMNO'].isin(universe_47)]
               .groupby('PERMNO')
               .agg(TICKER=('TICKER', 'last'),
                    COMNAM=('COMNAM', 'last'))
               .reset_index()
               .sort_values('PERMNO'))
print(f"\n47-stock universe (intersection of train and val full-coverage PERMNOs):")
print(universe_df.to_string(index=False))
